In [ ]:
import numpy as np
import gymnasium as gym
import random
import pandas as pd
env = gym.make("FrozenLake-v1")
alpha = 0.85
gamma = 0.90
epsilon = 0.8
num_episodes = 5000
num_timesteps = 1000
def random_policy():
    return env.action_space.sample()
v = {s: 0.0 for s in range(env.observation_space.n)}
for i in range(num_episodes):
    s, info = env.reset()
    for t in range(num_timesteps):
        a = random_policy()
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        v[s] += alpha * (r + gamma * v[s_] - v[s])
        s = s_
        if done:
            break
print("TD Prediction:")
print(pd.DataFrame(list(v.items()), columns=["State", "Value"]))
Q = {
    (s, a): 0.0
    for s in range(env.observation_space.n)
    for a in range(env.action_space.n)
}
def epsilon_greedy(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    return max(
        range(env.action_space.n),
        key=lambda a: Q[(state, a)]
    )
for i in range(num_episodes):
    s, info = env.reset()
    a = epsilon_greedy(s, epsilon)
    for t in range(num_timesteps):
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        a_ = epsilon_greedy(s_, epsilon)
        target = r if done else r + gamma * Q[(s_, a_)]
        Q[(s, a)] += alpha * (target - Q[(s, a)])
        s = s_
        a = a_
        if done:
            break
print("\nSARSA:")
print(pd.DataFrame(
    [(s, a, value) for (s, a), value in Q.items()],
    columns=["State", "Action", "Q-Value"]
))
for i in range(num_episodes):
    s, info = env.reset()
    for t in range(num_timesteps):
        a = epsilon_greedy(s, epsilon)
        s_, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        target = r if done else r + gamma * max(
            Q[(s_, next_a)] for next_a in range(env.action_space.n)
        )
        Q[(s, a)] += alpha * (target - Q[(s, a)])
        s = s_
        if done:
            break
print("\nQ-Learning:")
print(pd.DataFrame(
    [(s, a, value) for (s, a), value in Q.items()],
    columns=["State", "Action", "Q-Value"]
))
env.close()

TD Prediction:
    State         Value
0       0  3.973784e-05
1       1  1.467316e-05
2       2  7.594562e-07
3       3  2.743014e-08
4       4  6.437614e-06
5       5  0.000000e+00
6       6  4.405379e-06
7       7  0.000000e+00
8       8  9.183689e-04
9       9  2.824468e-02
10     10  4.305994e-03
11     11  0.000000e+00
12     12  0.000000e+00
13     13  3.766494e-02
14     14  8.554794e-01
15     15  0.000000e+00
